In [21]:
GET /api/health → {status: 'ok', db: 'connected'}
POST /api/auth/login → {access_token: '---', user: {...}}
GET /api/libros → [{id, titulo, disponible}, ...]
POST /api/prestamos → {prestamo_id, libro_id, fecha_devolucion}
POST /api/devoluciones → {multa: 0.0, estado: 'devuelto'}
GET  /api/multas/{estudiante} → {multa_total, prestamos_vencidos}

SyntaxError: invalid character '→' (U+2192) (1759312357.py, line 1)

In [24]:
import pytest
import requests

BASE = "http://localhost:8000/api"


@pytest.mark.smoke
class TestBiblotecaSmoke:

    def test_01_api_disponible(self):
        response = requests.get(f"{BASE}/health")
        assert response.status_code == 200
        assert response.json()['status'] == 'ok'

    def test_02_login_funciona(self):
        # Assuming a default login endpoint and credentials for a smoke test
        login_data = {"username": "testuser", "password": "testpassword"}
        response = requests.post(f"{BASE}/auth/login", json=login_data)
        assert response.status_code == 200
        assert 'access_token' in response.json()
        assert 'user' in response.json()

    def test_03_listar_libros_responde(self):
        response = requests.get(f"{BASE}/libros")
        assert response.status_code == 200
        assert isinstance(response.json(), list) # Expect a list of books

    def test_04_base_de_datos_conectada(self):
        response = requests.get(f"{BASE}/health")
        assert response.status_code == 200
        assert response.json()['db'] == 'connected'

    def test_05_modulo_multas_responde(self):
        # Assuming a default student ID for a smoke test on multas module
        response = requests.get(f"{BASE}/multas/some_student_id")
        assert response.status_code == 200
        assert 'multa_total' in response.json()
        assert 'prestamos_vencidos' in response.json()

In [5]:
# Create the directory structure and placeholder files for 'biblioteca' module
!mkdir -p biblioteca
!touch biblioteca/__init__.py
!touch biblioteca/prestamos.py
!touch biblioteca/multas.py

# Add dummy content to the files to allow successful imports and provide basic functionality for testing
# For prestamos.py
with open('biblioteca/prestamos.py', 'w') as f:
    f.write(
def calcular_disponibilidad(libro_id, estudiante_id=None):
    # Dummy implementation for now
    print(f"Dummy: Calculando disponibilidad para libro {libro_id}")
    return True

def registrar_prestamo(libro_id, estudiante_id, fecha_prestamo, fecha_devolucion_esperada):
    # Dummy implementation for now
    print(f"Dummy: Registrando préstamo de libro {libro_id} por estudiante {estudiante_id}")
    return {'prestamo_id': 1, 'libro_id': libro_id, 'fecha_devolucion': fecha_devolucion_esperada}

def registrar_devolucion(prestamo_id, fecha_devolucion_real):
    # Dummy implementation for now
    print(f"Dummy: Registrando devolución para préstamo {prestamo_id}")
    return {'multa': 0.0, 'estado': 'devuelto'}
)

# For multas.py
with open('biblioteca/multas.py', 'w') as f:
    f.write(
def calcular_multa(prestamo_id, fecha_devolucion_real, tarifa_diaria=500):
    # Dummy implementation for now
    print(f"Dummy: Calculando multa para préstamo {prestamo_id}")
    return 0.0
)

print("Dummy 'biblioteca' module files created successfully. You can now define your actual logic and tests.")

Dummy 'biblioteca' module files created successfully. You can now define your actual logic and tests.


In [3]:
# Create the directory structure and placeholder files for 'biblioteca' module
!mkdir -p biblioteca
!touch biblioteca/__init__.py
!touch biblioteca/prestamos.py
!touch biblioteca/multas.py

# Add dummy content to the files to allow successful imports and provide basic functionality for testing
# For prestamos.py
with open('biblioteca/prestamos.py', 'w') as f:
    f.write("""
def calcular_disponibilidad(libro_id, estudiante_id=None):
    # Dummy implementation for now
    print(f"Dummy: Calculando disponibilidad para libro {libro_id}")
    return True

def registrar_prestamo(libro_id, estudiante_id, fecha_prestamo, fecha_devolucion_esperada):
    # Dummy implementation for now
    print(f"Dummy: Registrando préstamo de libro {libro_id} por estudiante {estudiante_id}")
    return {'prestamo_id': 1, 'libro_id': libro_id, 'fecha_devolucion': fecha_devolucion_esperada}

def registrar_devolucion(prestamo_id, fecha_devolucion_real):
    # Dummy implementation for now
    print(f"Dummy: Registrando devolución para préstamo {prestamo_id}")
    return {'multa': 0.0, 'estado': 'devuelto'}
""")

# For multas.py
with open('biblioteca/multas.py', 'w') as f:
    f.write(
def calcular_multa(prestamo_id, fecha_devolucion_real, tarifa_diaria=500):
    # Dummy implementation for now
    print(f"Dummy: Calculando multa para préstamo {prestamo_id}")
    return 0.0
)

print("Dummy 'biblioteca' module files created successfully. You can now define your actual logic and tests.")

Dummy 'biblioteca' module files created successfully. You can now define your actual logic and tests.


### Updating `biblioteca/multas.py` with a function to consult student fines.

First, we need to update `biblioteca/multas.py` to include a function that can check if a student has any pending fines. This function will be called by `biblioteca/prestamos.py`.

In [20]:
from datetime import datetime, timedelta

with open('biblioteca/multas.py', 'w') as f:
    f.write(
from datetime import datetime, timedelta

# "Base de datos" en memoria para propósitos de demostración
# Idealmente, esto debería ser una base de datos real o un ORM en una aplicación real.
_multas_db = {
    "estudiante_A": {"multa_total": 500, "prestamos_vencidos": [101]}, # Ejemplo: el estudiante A tiene una multa
    "estudiante_B": {"multa_total": 0, "prestamos_vencidos": []},
    "estudiante_C": {"multa_total": 0, "prestamos_vencidos": []},
}

def calcular_multa(prestamo_id, fecha_devolucion_real, fecha_devolucion_esperada, tarifa_diaria=500):
    # Calcula la multa si la fecha de devolución real es posterior a la esperada
    if fecha_devolucion_real > fecha_devolucion_esperada:
        dias_retraso = (fecha_devolucion_real - fecha_devolucion_esperada).days
        return dias_retraso * tarifa_diaria
    return 0.0

def consultar_multas_estudiante(estudiante_id):
    # Consulta las multas pendientes de un estudiante
    return _multas_db.get(estudiante_id, {"multa_total": 0, "prestamos_vencidos": []})

def registrar_multa_estudiante(estudiante_id, multa_monto, prestamo_id=None):
    # Registra una multa para un estudiante
    if estudiante_id not in _multas_db:
        _multas_db[estudiante_id] = {"multa_total": 0, "prestamos_vencidos": []}
    _multas_db[estudiante_id]["multa_total"] += multa_monto
    if prestamo_id and prestamo_id not in _multas_db[estudiante_id]["prestamos_vencidos"]:
        _multas_db[estudiante_id]["prestamos_vencidos"].append(prestamo_id)
    return _multas_db[estudiante_id]["multa_total"]

def limpiar_multa_estudiante(estudiante_id):
    # Limpia todas las multas de un estudiante
    if estudiante_id in _multas_db:
        _multas_db[estudiante_id]["multa_total"] = 0
        _multas_db[estudiante_id]["prestamos_vencidos"] = []
    return 0.0

)
print("biblioteca/multas.py actualizado con lógica de consulta y registro.")

SyntaxError: invalid syntax (3783892080.py, line 5)

In [33]:
from datetime import datetime, timedelta

with open('biblioteca/prestamos.py', 'w') as f:
    f.write(
from datetime import datetime, timedelta
from biblioteca.multas import calcular_multa, consultar_multas_estudiante, registrar_multa_estudiante

# "Base de datos" en memoria para propósitos de demostración
# Idealmente, esto debería ser una base de datos real o un ORM en una aplicación real.
_libros_db = {
    1: {"id": 1, "titulo": "Cien años de soledad", "disponible": True},
    2: {"id": 2, "titulo": "El amor en los tiempos del cólera", "disponible": True},
    3: {"id": 3, "titulo": "La casa de los espíritus", "disponible": True},
    4: {"id": 4, "titulo": "Don Quijote de la Mancha", "disponible": True},
}

_prestamos_db = [
    {"prestamo_id": 101, "libro_id": 3, "estudiante_id": "estudiante_C",
     "fecha_prestamo": datetime(2024, 3, 1), "fecha_devolucion_esperada": datetime(2024, 3, 15),
     "fecha_devolucion_real": None, "multa_aplicada": 0.0}
]

_next_prestamo_id = 102 # Lleva el control del próximo ID de préstamo

def calcular_disponibilidad(libro_id, estudiante_id=None):
    # Obtiene la información del libro
    libro = _libros_db.get(libro_id)
    if not libro: # Si el libro no existe
        return False

    # Verifica si el libro está actualmente prestado
    for prestamo in _prestamos_db:
        if prestamo['libro_id'] == libro_id and prestamo['fecha_devolucion_real'] is None:
            return False # El libro está actualmente prestado

    # Si se proporciona un estudiante_id, verifica las multas pendientes
    if estudiante_id:
        multas_info = consultar_multas_estudiante(estudiante_id)
        if multas_info['multa_total'] > 0:
            # Según AC-3: Si el estudiante tiene multa > $0, no puede hacer nuevas reservas
            # Para simplificar, asumimos que 'disponibilidad' para un estudiante también significa que puede pedir prestado.
            # Esto se puede refinar para devolver 'True' pero bloquear el préstamo en 'registrar_prestamo'.
            # Por ahora, significa que no está disponible para este estudiante.
            return False

    return True # El libro está disponible y el estudiante puede pedirlo prestado (si aplica)

def registrar_prestamo(libro_id, estudiante_id, fecha_prestamo, fecha_devolucion_esperada):
    global _next_prestamo_id

    if not calcular_disponibilidad(libro_id, estudiante_id):
        return None # Libro no disponible o estudiante tiene multas

    prestamo = {
        'prestamo_id': _next_prestamo_id,
        'libro_id': libro_id,
        'estudiante_id': estudiante_id,
        'fecha_prestamo': fecha_prestamo,
        'fecha_devolucion_esperada': fecha_devolucion_esperada,
        'fecha_devolucion_real': None,
        'multa_aplicada': 0.0
    }
    _prestamos_db.append(prestamo)
    _libros_db[libro_id]['disponible'] = False # Marca el libro como no disponible
    _next_prestamo_id += 1
    return prestamo

def registrar_devolucion(prestamo_id, fecha_devolucion_real):
    for prestamo in _prestamos_db:
        if prestamo['prestamo_id'] == prestamo_id:
            if prestamo['fecha_devolucion_real'] is not None:
                return {'error': 'Préstamo ya devuelto'}

            prestamo['fecha_devolucion_real'] = fecha_devolucion_real
            _libros_db[prestamo['libro_id']]['disponible'] = True # Marca el libro como disponible

            multa = calcular_multa(
                prestamo_id,
                fecha_devolucion_real,
                prestamo['fecha_devolucion_esperada']
            )
            prestamo['multa_aplicada'] = multa

            # Registra la multa con el módulo de multas
            if multa > 0:
                registrar_multa_estudiante(prestamo['estudiante_id'], multa, prestamo_id)

            return {'multa': multa, 'estado': 'devuelto'}
    return {'error': 'Préstamo no encontrado'}
)
print("biblioteca/prestamos.py actualizado con lógica real.")

SyntaxError: invalid syntax (760875252.py, line 5)

In [32]:
import pytest
from biblioteca.prestamos import calcular_disponibilidad, registrar_prestamo, registrar_devolucion
from biblioteca.multas import calcular_multa

In [31]:
import sys
import os

# Add the current directory to sys.path to allow importing 'biblioteca'
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f"Current directory added to sys.path: {os.getcwd()}")

Current directory added to sys.path: /content


In [30]:
# Now, attempt the imports again
import pytest
from biblioteca.prestamos import calcular_disponibilidad, registrar_prestamo, registrar_devolucion
from biblioteca.multas import calcular_multa

print("Imports successful!")

Imports successful!


In [29]:
import pytest
from datetime import datetime, timedelta

# Necesitamos re-importar estos para obtener las versiones actualizadas después de las escrituras de archivos
from biblioteca.prestamos import calcular_disponibilidad, registrar_prestamo, registrar_devolucion, _prestamos_db, _libros_db
from biblioteca.multas import calcular_multa, consultar_multas_estudiante, registrar_multa_estudiante, limpiar_multa_estudiante, _multas_db

# Función de ayuda para reiniciar el estado de la "base de datos" antes de cada prueba
def reset_db_state():
    global _prestamos_db, _libros_db, _multas_db
    _libros_db.clear()
    _libros_db.update({
        1: {"id": 1, "titulo": "Cien años de soledad", "disponible": True},
        2: {"id": 2, "titulo": "El amor en los tiempos del cólera", "disponible": True},
        3: {"id": 3, "titulo": "La casa de los espíritus", "disponible": True},
        4: {"id": 4, "titulo": "Don Quijote de la Mancha", "disponible": True},
    })
    _prestamos_db.clear()
    _prestamos_db.append(
        {"prestamo_id": 101, "libro_id": 3, "estudiante_id": "estudiante_C",
         "fecha_prestamo": datetime(2024, 3, 1), "fecha_devolucion_esperada": datetime(2024, 3, 15),
         "fecha_devolucion_real": None, "multa_aplicada": 0.0}
    )
    _multas_db.clear()
    _multas_db.update({
        "estudiante_A": {"multa_total": 500, "prestamos_vencidos": [101]}, # Ejemplo: el estudiante A tiene una multa
        "estudiante_B": {"multa_total": 0, "prestamos_vencidos": []},
        "estudiante_C": {"multa_total": 0, "prestamos_vencidos": []},
    })

@pytest.fixture(autouse=True)
def setup_function():
    reset_db_state()
    yield

@pytest.mark.regression
class TestPrestamosRegression:

    def test_libro_prestado_no_esta_disponible(self):
        """
        REGRESION: Verifica que un libro actualmente prestado
        no aparece como disponible.
        """
        # El libro 3 está prestado a estudiante_C en el estado inicial
        assert not calcular_disponibilidad(3, "estudiante_X")
        assert not _libros_db[3]['disponible']

    def test_libro_disponible_tras_devolucion(self):
        """
        REGRESION: Verifica que el libro queda disponible
        inmediatamente después de la devolución.
        """
        # El libro 3 está prestado a estudiante_C inicialmente
        assert not calcular_disponibilidad(3, "estudiante_X")

        # Devuelve el libro
        devolucion_info = registrar_devolucion(101, datetime.now())
        assert devolucion_info['estado'] == 'devuelto'

        # El libro ahora debería estar disponible
        assert calcular_disponibilidad(3, "estudiante_X")
        assert _libros_db[3]['disponible']

    def test_bug_2024_031_no_disponible_con_prestamo_antiguo(self):
        """
        REGRESION de BUG-2024-031: Un libro prestado hace más de
        30 días NO debe aparecer como disponible.
        """
        # Simula un préstamo de hace 45 días para el libro 4 al estudiante_B
        fecha_prestamo_antigua = datetime.now() - timedelta(days=45)
        fecha_devolucion_esperada_antigua = fecha_prestamo_antigua + timedelta(days=15)

        # Agrega manualmente un préstamo antiguo a _prestamos_db (para un libro diferente al predeterminado de setup_function)
        _prestamos_db.append(
            {"prestamo_id": 102, "libro_id": 4, "estudiante_id": "estudiante_B",
             "fecha_prestamo": fecha_prestamo_antigua, "fecha_devolucion_esperada": fecha_devolucion_esperada_antigua,
             "fecha_devolucion_real": None, "multa_aplicada": 0.0}
        )
        _libros_db[4]['disponible'] = False

        # El libro 4 no debería estar disponible, incluso si el préstamo es antiguo
        assert not calcular_disponibilidad(4, "estudiante_X")

    def test_multa_no_afecta_disponibilidad_de_otro_libro(self):
        """
        NUEVO con merge: Tener una multa no debe cambiar la
        disponibilidad de libros que no están prestados.
        """
        # el estudiante_A tiene una multa inicialmente (500)
        # El libro 1 está disponible inicialmente y no está relacionado con la multa del estudiante_A
        assert _libros_db[1]['disponible']

        # Verifica la disponibilidad del Libro 1 para un estudiante SIN multas (estudiante_B)
        assert calcular_disponibilidad(1, "estudiante_B")

        # Verifica la disponibilidad del Libro 1 para un estudiante CON multas (estudiante_A)
        # Debería devolver False porque el estudiante_A tiene multas, según AC-3
        assert not calcular_disponibilidad(1, "estudiante_A")

        # Asegura que el libro en sí todavía se considera disponible para otros estudiantes o verificación general
        # La bandera 'disponible' en el objeto del libro solo refleja si está actualmente prestado.
        # La función `calcular_disponibilidad` verifica tanto la disponibilidad del libro como el estado de multa del estudiante.
        assert _libros_db[1]['disponible'] is True


    def test_calculo_multa_tres_dias_retraso(self):
        """
        REGRESION: La multa por 3 días de retraso debe ser
        exactamente 3 * tarifa_diaria (configurada en settings).
        """
        tarifa_diaria = 500 # Esto está codificado en biblioteca/multas.py por ahora

        # Simula un préstamo con 3 días de retraso
        prestamo_id_test = 200
        estudiante_id_test = "estudiante_test_multa"
        libro_id_test = 2

        fecha_prestamo = datetime(2024, 3, 10)
        fecha_devolucion_esperada = datetime(2024, 3, 20)
        fecha_devolucion_real = datetime(2024, 3, 23) # 3 días tarde

        # Registra un nuevo préstamo primero para que entre en el sistema (este para el libro 2)
        registrar_prestamo(libro_id_test, estudiante_id_test, fecha_prestamo, fecha_devolucion_esperada)

        # Encuentra el prestamo_id real generado (será _next_prestamo_id - 1)
        # Para simplificar, podemos usar el siguiente ID disponible de _prestamos_db, asumiendo que no hay otras pruebas ejecutándose concurrentemente.
        # O, podemos encontrar directamente el último préstamo agregado.
        actual_prestamo = next((p for p in _prestamos_db if p['libro_id'] == libro_id_test and p['estudiante_id'] == estudiante_id_test and p['fecha_devolucion_real'] is None), None)
        assert actual_prestamo is not None
        test_prestamo_id = actual_prestamo['prestamo_id']

        # Registra la devolución con retraso
        devolucion_info = registrar_devolucion(test_prestamo_id, fecha_devolucion_real)

        expected_multa = 3 * tarifa_diaria
        assert devolucion_info['multa'] == expected_multa

        # Verifica que la multa esté registrada con el estudiante
        multas_estudiante = consultar_multas_estudiante(estudiante_id_test)
        assert multas_estudiante['multa_total'] == expected_multa
        assert test_prestamo_id in multas_estudiante['prestamos_vencidos']

### Implementando Pruebas de Aceptación (Acceptance Testing)

Ahora, implementaremos los escenarios Gherkin para la historia de usuario US-015 y sus criterios de aceptación.

In [18]:
# Create the 'features' directory and the 'multas.feature' file
!mkdir -p features

with open('features/multas.feature', 'w') as f:
    f.write(
Feature: Sistema de multas por devolución tardía
  Como estudiante de la Universitaria de Colombia
  Quiero conocer y gestionar mis multas de biblioteca
  Para poder planificar mis reservas de libros

  # AC-1: Cálculo correcto de la multa
  Scenario: Cálculo de multa por 3 días de retraso
    Given que el libro "El Quijote" fue prestado al "Estudiante A" el "2024-03-01"
    And que la fecha de devolución esperada era el "2024-03-15"
    When el "Estudiante A" devuelve el libro "El Quijote" el "2024-03-18"
    Then la multa calculada debe ser de "1500" COP
    And el "Estudiante A" debe tener una multa total de "1500" COP

  # AC-2: Mensaje cuando no hay multa
  Scenario: Estudiante sin multa pendiente
    Given que el "Estudiante B" no tiene multas pendientes
    When consulto el estado de multas del "Estudiante B"
    Then el mensaje debe indicar 'Sin multa pendiente'
    And la multa total debe ser de "0" COP

  # AC-3: Bloqueo por multa pendiente
  Scenario: Estudiante con multa no puede reservar
    Given que el "Estudiante C" tiene una multa pendiente de "500" COP
    And que el libro "Cien años de soledad" está disponible
    When el "Estudiante C" intenta registrar un préstamo del libro "Cien años de soledad"
    Then el préstamo no debe ser registrado
    And el libro "Cien años de soledad" debe seguir disponible
)
print("features/multas.feature file created successfully.")

features/multas.feature file created successfully.
